loading + Step 1-5 er merge code

In [1]:
# SETUP — Re-create the merged dataset (Steps 1–5 from notebook 01)

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

folder = "/content/drive/MyDrive/Project2_SolarPV/"

# Step 1-2: Load Plant 1 files (already selected — data-driven decision)

gen = pd.read_csv(folder + "Plant_1_Generation_Data.csv")
wth = pd.read_csv(folder + "Plant_1_Weather_Sensor_Data.csv")

# Correct explicit date parsing (verified formats from Step 1)
gen["DATE_TIME"] = pd.to_datetime(gen["DATE_TIME"], format="%d-%m-%Y %H:%M")
wth["DATE_TIME"] = pd.to_datetime(wth["DATE_TIME"], format="%Y-%m-%d %H:%M:%S")

# Step 3: Plant-level aggregation (22 inverters -> 1 plant row)

gen_plant = gen.groupby("DATE_TIME").agg({
    "DC_POWER": "sum",
    "AC_POWER": "sum",
    "DAILY_YIELD": "sum",
    "TOTAL_YIELD": "sum",
}).reset_index()

# Step 4: Merge with weather data (inner join)

wth_clean = wth[["DATE_TIME", "AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]].copy()
merged = pd.merge(gen_plant, wth_clean, on="DATE_TIME", how="inner")

# Sanity check — confirm it matches notebook 01's numbers

print("Merged shape:", merged.shape)          # expect (3157, 8)
print("Missing values:\n", merged.isna().sum())
print("Date range:", merged["DATE_TIME"].min(), "→", merged["DATE_TIME"].max())

Mounted at /content/drive
Merged shape: (3157, 8)
Missing values:
 DATE_TIME              0
DC_POWER               0
AC_POWER               0
DAILY_YIELD            0
TOTAL_YIELD            0
AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
dtype: int64
Date range: 2020-05-15 00:00:00 → 2020-06-17 23:45:00


got missing valus 0, Merged shape: (3157, 8)

In [2]:
# CLEANING AND FEATURE ENGINEERING

df = merged.copy()
df = df.sort_values("DATE_TIME").reset_index(drop=True)

# Drop columns not needed for modeling
cols_to_drop = ["DAILY_YIELD", "TOTAL_YIELD", "DC_POWER", "PLANT_ID", "SOURCE_KEY"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print("Columns after drop:", df.columns.tolist())

# Identify timestamp gaps
expected_gap = pd.Timedelta(minutes=15)
df["time_diff"] = df["DATE_TIME"].diff()
df["gap_before"] = df["time_diff"] != expected_gap
df.loc[0, "gap_before"] = False

n_gaps = df["gap_before"].sum()
print(f"Number of rows immediately AFTER a gap: {n_gaps}")
print(df.loc[df["gap_before"], ["DATE_TIME", "time_diff"]])

# Construct the TARGET: AC_POWER(t+1)
df["TARGET_AC_POWER_t_plus1"] = df["AC_POWER"].shift(-1)
df["target_gap_valid"] = ~df["gap_before"].shift(-1).fillna(True)

# Construct LAG features for Model B
df["AC_POWER_lag1"] = df["AC_POWER"].shift(1)
df["AC_POWER_lag2"] = df["AC_POWER"].shift(2)
df["AC_POWER_lag3"] = df["AC_POWER"].shift(3)

df["lags_valid"] = (
    (~df["gap_before"]) &
    (~df["gap_before"].shift(1).fillna(True)) &
    (~df["gap_before"].shift(2).fillna(True)) &
    (~df["gap_before"].shift(3).fillna(True))
)

# Drop rows that are NOT usable
before_drop = len(df)

df_clean = df[
    df["target_gap_valid"] &
    df["lags_valid"] &
    df["TARGET_AC_POWER_t_plus1"].notna()
].copy()

after_drop = len(df_clean)
print(f"Rows before cleaning: {before_drop}")
print(f"Rows after cleaning: {after_drop}")
print(f"Rows dropped: {before_drop - after_drop}")

# Drop helper columns
df_clean = df_clean.drop(columns=["time_diff", "gap_before", "target_gap_valid", "lags_valid"])

print("\nFinal cleaned dataset shape:", df_clean.shape)
print("Final columns:", df_clean.columns.tolist())
df_clean.head()

Columns after drop: ['DATE_TIME', 'AC_POWER', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
Number of rows immediately AFTER a gap: 10
               DATE_TIME       time_diff
93   2020-05-16 02:00:00 0 days 03:00:00
420  2020-05-19 12:30:00 0 days 01:00:00
520  2020-05-20 17:30:00 0 days 04:15:00
542  2020-05-21 07:45:00 0 days 09:00:00
724  2020-05-23 06:45:00 0 days 01:45:00
784  2020-05-23 22:00:00 0 days 00:30:00
911  2020-05-25 06:00:00 0 days 00:30:00
1265 2020-05-29 06:15:00 0 days 08:00:00
1776 2020-06-03 14:15:00 0 days 00:30:00
3088 2020-06-17 06:45:00 0 days 00:45:00
Rows before cleaning: 3157
Rows after cleaning: 3103
Rows dropped: 54

Final cleaned dataset shape: (3103, 9)
Final columns: ['DATE_TIME', 'AC_POWER', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'TARGET_AC_POWER_t_plus1', 'AC_POWER_lag1', 'AC_POWER_lag2', 'AC_POWER_lag3']


/tmp/ipykernel_1576/3922646427.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["target_gap_valid"] = ~df["gap_before"].shift(-1).fillna(True)
/tmp/ipykernel_1576/3922646427.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (~df["gap_before"].shift(1).fillna(True)) &
/tmp/ipykernel_1576/3922646427.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_d

,DATE_TIME,AC_POWER,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,TARGET_AC_POWER_t_plus1,AC_POWER_lag1,AC_POWER_lag2,AC_POWER_lag3
3,2020-05-15 00:45:00,0.0,24.846130,22.360852,0.0,0.0,0.0,0.0,0.0
4,2020-05-15 01:00:00,0.0,24.621525,22.165423,0.0,0.0,0.0,0.0,0.0
5,2020-05-15 01:15:00,0.0,24.536092,21.968571,0.0,0.0,0.0,0.0,0.0
6,2020-05-15 01:30:00,0.0,24.638674,22.352926,0.0,0.0,0.0,0.0,0.0
7,2020-05-15 01:45:00,0.0,24.873022,23.160919,0.0,0.0,0.0,0.0,0.0


**Timestamp gap detect**| WHY: Data is expected at 15-minute intervals, but there are gaps in some places (due to sensor downtime, etc.). If these gaps are not identified, the target and lag features might be derived from incorrect points.

In [3]:
# Detect timestamp gaps
expected_gap = pd.Timedelta(minutes=15)
df["time_diff"] = df["DATE_TIME"].diff()

# gap_before = True mane: ei row-er age-er row theke thik 15 min gap na
df["gap_before"] = df["time_diff"] != expected_gap
df.loc[0, "gap_before"] = False   # first row-er age kono row nei, tai False

n_gaps = df["gap_before"].sum()
print(f"Number of rows immediately AFTER a gap: {n_gaps}")
print(df.loc[df["gap_before"], ["DATE_TIME", "time_diff"]])

Number of rows immediately AFTER a gap: 10
               DATE_TIME       time_diff
93   2020-05-16 02:00:00 0 days 03:00:00
420  2020-05-19 12:30:00 0 days 01:00:00
520  2020-05-20 17:30:00 0 days 04:15:00
542  2020-05-21 07:45:00 0 days 09:00:00
724  2020-05-23 06:45:00 0 days 01:45:00
784  2020-05-23 22:00:00 0 days 00:30:00
911  2020-05-25 06:00:00 0 days 00:30:00
1265 2020-05-29 06:15:00 0 days 08:00:00
1776 2020-06-03 14:15:00 0 days 00:30:00
3088 2020-06-17 06:45:00 0 days 00:45:00


Target & lag feature making | WHY — Target: We want to predict what the AC_POWER will be in 15 minutes. Therefore, we set the "AC_POWER of the next 15 minutes" as the target in each row (using `shift(-1)`).

WHY — Lag: For the model, we need the past three AC_POWER values ​​as features (using `shift(1)`, `shift(2)`, and `shift(3)`).

WHY — Validity check: If there is a gap, the "next row" or "previous row" does not represent a 15-minute interval—it corresponds to a different timeframe. Thus, if a gap exists, we mark that target/lag as invalid.

In [5]:
# Construct target and lag features (warning-free)

# Target: AC_POWER 15 minutes ahead
df["TARGET_AC_POWER_t_plus1"] = df["AC_POWER"].shift(-1)

# Is the (t -> t+1) step a real 15-min step? (no gap in between)
df["target_gap_valid"] = ~df["gap_before"].shift(-1, fill_value=True)

# Lag features: past 3 AC_POWER values
df["AC_POWER_lag1"] = df["AC_POWER"].shift(1)
df["AC_POWER_lag2"] = df["AC_POWER"].shift(2)
df["AC_POWER_lag3"] = df["AC_POWER"].shift(3)

# Are all 3 lag steps real 15-min steps? (no gap anywhere in the lag window)
df["lags_valid"] = (
    (~df["gap_before"]) &
    (~df["gap_before"].shift(1, fill_value=True)) &
    (~df["gap_before"].shift(2, fill_value=True)) &
    (~df["gap_before"].shift(3, fill_value=True))
)

print("Target and lag columns created. Checking for warnings above this line...")

Target and lag columns created. Checking for warnings above this line...


**Invalid rows drop** | Here's why: Rows where the target or label is invalid (gap-affected) cannot be used for model training—otherwise, the model would learn incorrect patterns. That is why they are dropped.

In [6]:
# Drop invalid rows
before_drop = len(df)

df_clean = df[
    df["target_gap_valid"] &
    df["lags_valid"] &
    df["TARGET_AC_POWER_t_plus1"].notna()   # last row-er target NaN thakbe, oita bad
].copy()

after_drop = len(df_clean)
print(f"Rows before cleaning: {before_drop}")
print(f"Rows after cleaning: {after_drop}")
print(f"Rows dropped: {before_drop - after_drop}")

Rows before cleaning: 3157
Rows after cleaning: 3103
Rows dropped: 54


**Helper columns clean** | Reasons: time_gap, gap_before, etc. These were created solely for calculation purposes; the model doesn't actually need them. I drop them to keep the final dataset clean.

In [7]:
# drop helper clean
df_clean = df_clean.drop(columns=["time_diff", "gap_before", "target_gap_valid", "lags_valid"])

print("Final cleaned dataset shape:", df_clean.shape)
print("Final columns:", df_clean.columns.tolist())
df_clean.head()

Final cleaned dataset shape: (3103, 9)
Final columns: ['DATE_TIME', 'AC_POWER', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'TARGET_AC_POWER_t_plus1', 'AC_POWER_lag1', 'AC_POWER_lag2', 'AC_POWER_lag3']


,DATE_TIME,AC_POWER,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,TARGET_AC_POWER_t_plus1,AC_POWER_lag1,AC_POWER_lag2,AC_POWER_lag3
3,2020-05-15 00:45:00,0.0,24.846130,22.360852,0.0,0.0,0.0,0.0,0.0
4,2020-05-15 01:00:00,0.0,24.621525,22.165423,0.0,0.0,0.0,0.0,0.0
5,2020-05-15 01:15:00,0.0,24.536092,21.968571,0.0,0.0,0.0,0.0,0.0
6,2020-05-15 01:30:00,0.0,24.638674,22.352926,0.0,0.0,0.0,0.0,0.0
7,2020-05-15 01:45:00,0.0,24.873022,23.160919,0.0,0.0,0.0,0.0,0.0


In [8]:

# PART 7 — Verification (optional, already confirmed correct)

lookup = df.set_index("DATE_TIME")["AC_POWER"]
check = df_clean.copy()

check["expected_target_by_time"] = check["DATE_TIME"].apply(
    lambda t: lookup.get(t + pd.Timedelta(minutes=15), np.nan)
)
print("TRUE target mismatches:", len(check[check["TARGET_AC_POWER_t_plus1"] != check["expected_target_by_time"]]))

check["expected_lag1_by_time"] = check["DATE_TIME"].apply(
    lambda t: lookup.get(t - pd.Timedelta(minutes=15), np.nan)
)
print("TRUE lag1 mismatches:", len(check[check["AC_POWER_lag1"] != check["expected_lag1_by_time"]]))

TRUE target mismatches: 0
TRUE lag1 mismatches: 0


In [9]:
df_clean.to_csv(folder + "df_clean_step6.csv", index=False)
print("Saved:", df_clean.shape)

Saved: (3103, 9)
